In [0]:
WITH concessions AS (
  SELECT report_month, population_type, 'Popcorn' AS category,
         concession_popcorn_revenue AS revenue,
         concession_popcorn_transactions AS transactions,
         concession_popcorn_quantity AS quantity,
         concession_popcorn_members AS members,
         ROUND(concession_popcorn_quantity * 1.0 / NULLIF(concession_popcorn_members, 0), 2) AS avg_item_per_member
  FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly

  UNION ALL
  SELECT report_month, population_type, 'Fountain Drinks',
         concession_fountain_drinks_revenue,
         concession_fountain_drinks_transactions,
         concession_fountain_drinks_quantity,
         concession_fountain_drinks_members,
         ROUND(concession_fountain_drinks_quantity * 1.0 / NULLIF(concession_fountain_drinks_members, 0), 2)
  FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly

  UNION ALL
  SELECT report_month, population_type, 'Candy',
         concession_candy_revenue,
         concession_candy_transactions,
         concession_candy_quantity,
         concession_candy_members,
         ROUND(concession_candy_quantity * 1.0 / NULLIF(concession_candy_members, 0), 2)
  FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly

  UNION ALL
  SELECT report_month, population_type, 'Hot Foods',
         concession_hot_foods_revenue,
         concession_hot_foods_transactions,
         concession_hot_foods_quantity,
         concession_hot_foods_members,
         ROUND(concession_hot_foods_quantity * 1.0 / NULLIF(concession_hot_foods_members, 0), 2)
  FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly

  UNION ALL
  SELECT report_month, population_type, 'Merch',
         concession_merch_revenue,
         concession_merch_transactions,
         concession_merch_quantity,
         concession_merch_members,
         ROUND(concession_merch_quantity * 1.0 / NULLIF(concession_merch_members, 0), 2)
  FROM data_exploration_hub.marketing_hub.mkt_dashboard_concessions_monthly
),
visit_grain AS (
  -- One row per visit: CDE_ID x date x location, refunds netted out
  SELECT t.CDE_ID, d.datefull, t.LocationId,
         MAX(CASE WHEN t.CineClubFLAG = 'Yes' THEN 1 ELSE 0 END) AS is_cc,
         MAX(CASE WHEN t.SCENEFLAG = 'Yes' THEN 1 ELSE 0 END) AS is_scene
  FROM cpx_dataanalytics_gold.customer360.c360_f_transaction t
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_date d
    ON t.VisitDateId = d.dateid
  JOIN cpx_dataanalytics_gold.customer360.cpx_d_location l
    ON t.LocationId = l.LocationID
  WHERE l.LC_Location_Type_Description = 'Cineplex Theatre'
    AND t.RecRoomFLAG = 'No'
    AND t.STOREFLAG = 'No'
    AND t.CDE_ID IS NOT NULL AND t.CDE_ID > 0
    AND (t.TicketFLAG = 'Yes' OR t.ConcessionFLAG = 'Yes')
    AND t.VisitDateId >= 20240101
  GROUP BY t.CDE_ID, d.datefull, t.LocationId
  HAVING SUM(t.Quantity) > 0
),
visits AS (
  SELECT TRUNC(datefull, 'MM') AS report_month,
         CASE WHEN is_cc = 1 THEN 'CC' ELSE 'Scene' END AS population_type,
         COUNT(*) AS visits,
         COUNT(DISTINCT CDE_ID) AS visiting_members
  FROM visit_grain
  WHERE is_cc = 1 OR is_scene = 1
  GROUP BY 1, 2
)
SELECT c.*, v.visits, v.visiting_members
FROM concessions c
LEFT JOIN visits v
  ON c.report_month = v.report_month
 AND c.population_type = v.population_type
WHERE c.report_month >= '2024-01-01'
ORDER BY c.report_month, c.population_type, c.category;